# 02 — Modeling

Reports the v3.0 ship-Model-A outcome per spec §8 + §9.2 + §15.6.

> **v3.0 update (2026-06-09):** the project's original framing ("Model B should beat Model A by adding the SA2 augmentor block") returned a null result under proper 6-fold k-fold validation. The 8 augmentor variants tested in Phase 2 produced 0 robust wins. Phase 2.5 postmortem (rank consistency + seed-noise floor + explicit interaction probe + Optuna hyperparameter sweep) confirmed: ship Model A on tuned hyperparameters. Full evidence: `docs/research/2026-06_v3.0_phase3_closing_summary.md`.
>
> This notebook leads with Model A on the tuned defaults and reports v3.0 k-fold metrics as the headline. Model B is loaded for historical comparison only (the artifacts still exist in `models/` because `make train` builds them for reproducibility of the v2.x story).

**Design note.** Spec §9 states *"none of [the notebooks] refit data or re-call APIs."* The actual fitting lives in `fuel_pred.train.train_models` (run via `make train`) and `fuel_pred.train.cv` (run via `make train-kfold`). This notebook **loads** those artifacts and presents the modeling analysis — it does not refit.

> Run `make train && make evaluate && make train-kfold && make evaluate-kfold` first if any artifacts aren't present.


## Setup

In [ ]:
from __future__ import annotations

import datetime as dt
import json
import pickle
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from fuel_pred import config


def _require(path: Path) -> Path:
    if not path.exists():
        raise SystemExit(f"{path} not found — run `make train` (+ `make evaluate`) first.")
    return path


features = pd.read_parquet(_require(config.DATA_PROCESSED / "features.parquet"))
features["date"] = pd.to_datetime(features["date"])

with open(_require(config.MODELS_DIR / "model_a.pkl"), "rb") as fh:
    model_a = pickle.load(fh)
with open(_require(config.MODELS_DIR / "model_b.pkl"), "rb") as fh:
    model_b = pickle.load(fh)
with open(_require(config.MODELS_DIR / "feature_lists.json")) as fh:
    feature_lists = json.load(fh)

preds = {
    "test_normal": pd.read_parquet(_require(config.MODELS_DIR / "predictions_test_normal.parquet")),
    "test_crisis": pd.read_parquet(_require(config.MODELS_DIR / "predictions_test_crisis.parquet")),
}
for _df in preds.values():
    _df["date"] = pd.to_datetime(_df["date"])

print(f"features: {len(features):,} rows × {len(features.columns)} cols")
print(f"Model A: {model_a.n_features_in_} features; Model B: {model_b.n_features_in_} features")
for _name, _df in preds.items():
    print(f"{_name}: {len(_df):,} prediction rows")

## 1. Folds — v3.0 k-fold methodology

v3.0 replaced the v2.x single-split (train ≤ 2022, val 2023, test_normal 2024-25, test_crisis 2026) with 6-fold expanding-window k-fold CV across the full panel (spec §15.2). The v2.x single-split fold boundaries are loaded below for the historical comparison; the v3.0 k-fold geometry is documented in `results/v3_phase2_pr_b_baseline_kfold.md`.


In [ ]:
# Fold row counts (time-based, no shuffling — spec §8.3). The model trains on
# U91 rows with a non-null t+1 target; the second column reflects that subset.
FOLDS = [
    ("train",       config.SPAN_START,        config.TRAIN_END),
    ("val",         config.VAL_START,         config.VAL_END),
    ("test_normal", config.TEST_START,        config.TEST_NORMAL_END),
    ("test_crisis", config.TEST_CRISIS_START, features["date"].max().date().isoformat()),
]


def fold_label(d: pd.Timestamp) -> str:
    dd = d.date()
    for name, start, end in FOLDS:
        if dt.date.fromisoformat(start) <= dd <= dt.date.fromisoformat(end):
            return name
    return "out_of_span"


features["fold"] = features["date"].map(fold_label)
u91 = features[(features["fuel_code"] == "U91") & features["y_t1"].notna()]
pd.DataFrame({
    "all_rows": features["fold"].value_counts(),
    "U91_with_target": u91["fold"].value_counts(),
}).reindex([f[0] for f in FOLDS]).fillna(0).astype(int)

## 2. Feature columns

Model A: lag, upstream, calendar, ctx, stn, wx (no `sa2_*`). **Production model.**
Model B: same plus `sa2_*`. **Retired in v3.0** — kept as a buildable artifact for reproducibility.

The `sa2_*` block is still ingested into `features.parquet` (spec §7.7 — research surface), but the production Model A does not consume it. Under v3.0 k-fold validation no augmentor variant produces a robust lift over Model A; see Phase 3 closing summary doc for the full evidence.

**Identical training rows** for both A and B — only rows where every `sa2_*` column is non-null. This makes the historical A vs B comparison apples-to-apples (preserves spec §8.4's original framing for the v2.x record).


In [ ]:
cols_a = feature_lists["A"]["feature_columns"]
cols_b = feature_lists["B"]["feature_columns"]
sa2_in_b = [c for c in cols_b if c.startswith("sa2_")]
print(f"Model A: {len(cols_a)} feature columns (no sa2_*)")
print(f"Model B: {len(cols_b)} feature columns, of which {len(sa2_in_b)} are sa2_*")
print("\nThe SA2 block — the only difference between A and B:")
for c in sa2_in_b:
    print(f"  {c}")

# Identical-rows guard: both models train only where every sa2_* column is
# non-null, so the comparison isolates the SA2 block. Show survival in train.
train_u91 = features[(features["fuel_code"] == "U91")
                     & features["y_t1"].notna()
                     & (features["fold"] == "train")]
mask = train_u91[sa2_in_b].notna().all(axis=1)
print(f"\nidentical-rows guard (train fold): {int(mask.sum()):,} / {len(train_u91):,} "
      f"({100 * mask.mean():.1f}%) rows have all {len(sa2_in_b)} sa2_* non-null")

## 3. Model A — production (loaded)

**Production model.** Lag, upstream, calendar, ctx, stn, wx blocks; no `sa2_*`. Hyperparameters per spec §8.2 (v3.0-tuned via Optuna TPE in Phase 3 #4; validated across 6 seeds with a mean improvement of 0.170 c/L over the original v1/v2 defaults).


In [ ]:
# Model A summary + the shared hyperparameters (identical for both models).
print("Shared hyperparameters (spec §8.2):")
for k, v in config.LGBM_PARAMS.items():
    print(f"  {k}: {v}")
print()
print(f"Model A: {model_a.n_features_in_} features, "
      f"best_iteration={getattr(model_a, 'best_iteration_', 'n/a')}")

## 4. Model B — historical / retired (loaded)

**Retired in v3.0.** Same hyperparameters and training rows as Model A; the only addition was the SA2 augmentor block. Under v3.0 6-fold k-fold validation, Model B did *not* robustly outperform Model A across 8 augmentor variants tested. The artifacts persist for reproducibility of the v2.x experimental record.

Top gain-importances side by side below — useful for understanding which SA2 features the model *would* split on if it were in production. The v3.0 verdict isn't "the model ignored these features" but "the splits don't generalise across folds" (see `docs/research/2026-06_v3.0_phase3_closing_summary.md`, §3 interaction-probe section).


In [ ]:
print(f"Model B: {model_b.n_features_in_} features, "
      f"best_iteration={getattr(model_b, 'best_iteration_', 'n/a')}")
print(f"  = Model A's {model_a.n_features_in_} + "
      f"{model_b.n_features_in_ - model_a.n_features_in_} sa2_* features")


def _gain_imp(model, cols: list[str]) -> pd.Series:
    g = model.booster_.feature_importance(importance_type="gain")
    return pd.Series(g, index=cols).sort_values(ascending=False)


imp_a = _gain_imp(model_a, cols_a)
imp_b = _gain_imp(model_b, cols_b)
# reset_index() on an unnamed Series gives columns ["index", 0]; rename explicitly
# (the `names=` kwarg was not consistently supported across pandas versions).
top10 = pd.concat([
    imp_a.head(10).round(0).reset_index().rename(columns={"index": "A_feature", 0: "A_gain"}),
    imp_b.head(10).round(0).reset_index().rename(columns={"index": "B_feature", 0: "B_gain"}),
], axis=1)
top10


## 5. Headline metrics

**v3.0 headline: k-fold (6 folds, mean ± stdev) — see new cell below.**

The v2.x single-split numbers (test_normal + test_crisis) are still computed below for historical continuity, but they no longer drive the ship decision — v3.0 §15.2 deprecated the single-split scheme. MAE / RMSE / MAPE / median / p90 absolute error.


In [ ]:
from IPython.display import display


def _metrics(y_true: pd.Series, y_pred: pd.Series) -> dict[str, float]:
    err = y_pred - y_true
    ae = err.abs()
    mape = (ae / y_true.abs()).replace([np.inf, -np.inf], np.nan).mean() * 100
    return {
        "MAE": ae.mean(),
        "RMSE": float(np.sqrt((err**2).mean())),
        "MAPE_%": mape,
        "median_AE": float(ae.median()),
        "p90_AE": float(np.percentile(ae, 90)),
    }


rows = []
for fold, df in preds.items():
    for label, col in (("A", "y_pred_a"), ("B", "y_pred_b")):
        rows.append({"fold": fold, "model": label,
                     **{k: round(v, 3) for k, v in _metrics(df["y_true"], df[col]).items()}})
print("Per-model metrics:")
display(pd.DataFrame(rows).set_index(["fold", "model"]))

# Headline Δ (B − A); negative = Model B better.
delta = []
for fold, df in preds.items():
    mae_a = (df["y_pred_a"] - df["y_true"]).abs().mean()
    mae_b = (df["y_pred_b"] - df["y_true"]).abs().mean()
    delta.append({"fold": fold, "MAE_A": round(mae_a, 3), "MAE_B": round(mae_b, 3),
                  "Δ_MAE": round(mae_b - mae_a, 3), "rel_%": round(100 * (mae_b - mae_a) / mae_a, 2)})
print("\nHeadline Δ MAE (negative = augmentor adds value):")
pd.DataFrame(delta).set_index("fold")

## 5b. v3.0 headline — k-fold metrics (canonical)

The v3.0 canonical headline. PR B baseline (the same model config that v2.x shipped) under 6-fold time-series k-fold. Loaded from the published metrics JSON. **Mean Δ MAE is well inside the per-fold Stdev → Model B has no robust win.**


In [ ]:
# Load v3.0 metrics from the published JSON.
phase2_metrics_path = Path("results/v3_phase2_metrics.json")
if not phase2_metrics_path.exists():
    print(f"missing {phase2_metrics_path} — run tools/research/v3_phase2_kfold_runner.py")
else:
    phase2 = json.loads(phase2_metrics_path.read_text(encoding="utf-8"))
    summary_rows = []
    for exp_name, m in phase2.items():
        if "mean_delta_mae" not in m:
            continue
        mean_d = m["mean_delta_mae"]
        std_d = m["stdev_delta_mae"]
        if abs(mean_d) > 2 * std_d:
            verdict = "robust" if mean_d < 0 else "robust (B loses)"
        elif abs(mean_d) > std_d:
            verdict = "weak"
        else:
            verdict = "noise"
        summary_rows.append({
            "experiment": exp_name,
            "Mean Δ MAE (B−A)": round(mean_d, 3),
            "Stdev Δ MAE": round(std_d, 3),
            "Min Δ MAE": round(m["min_delta_mae"], 3),
            "Max Δ MAE": round(m["max_delta_mae"], 3),
            "verdict": verdict,
        })
    v3_summary_df = pd.DataFrame(summary_rows).set_index("experiment")
    display(v3_summary_df)

# v3.0 Phase 3 #4 (hyperopt) — Model A retune validation
hyperopt_val_path = Path("results/v3_phase3_hyperopt_validation.json")
if hyperopt_val_path.exists():
    val = json.loads(hyperopt_val_path.read_text(encoding="utf-8"))
    print()
    print(f"Phase 3 #4 hyperopt validation (Model A new defaults vs v1/v2):")
    print(f"  Mean improvement across folds: {val['mean_improvement_across_folds']:+.4f} c/L")
    print(f"  Stdev improvement across folds: {val['stdev_improvement_across_folds']:.4f} c/L")
    ratio = abs(val['mean_improvement_across_folds']) / val['stdev_improvement_across_folds']
    print(f"  Ratio |mean|/stdev: {ratio:.2f}")
    print(f"  Verdict: {'ROBUST' if val['verdict']['robust'] else 'WEAK' if val['verdict']['validated_significance'] else 'MARGINAL'}")


## 6. Segmented metrics

By metro/regional, brand, fuel type, SEIFA quintile.

In [ ]:
# Segmented Δ MAE on test_normal. Brand / metro / SEIFA keys aren't in the
# prediction parquets, so join them back from the feature slice.
seg_keys = [c for c in ("station_id", "date", "stn_brand_canonical",
                        "stn_is_metro", "sa2_seifa_irsd_score") if c in features.columns]
seg_slice = features[seg_keys].drop_duplicates(["station_id", "date"])
tn = preds["test_normal"].merge(seg_slice, on=["station_id", "date"], how="left")


def _seg_table(df: pd.DataFrame, group) -> pd.DataFrame:
    d = df.copy()
    d["ae_a"] = (d["y_pred_a"] - d["y_true"]).abs()
    d["ae_b"] = (d["y_pred_b"] - d["y_true"]).abs()
    t = d.groupby(group, observed=True).agg(
        n=("y_true", "size"), MAE_A=("ae_a", "mean"), MAE_B=("ae_b", "mean"))
    t["Δ_MAE"] = t["MAE_B"] - t["MAE_A"]
    return t.round(3)


# SEIFA quintile
try:
    tn["seifa_q"] = pd.qcut(tn["sa2_seifa_irsd_score"], 5,
                            labels=["Q1", "Q2", "Q3", "Q4", "Q5"], duplicates="drop")
except ValueError:
    tn["seifa_q"] = pd.qcut(tn["sa2_seifa_irsd_score"], 5, duplicates="drop")
print("test_normal — Δ MAE by SEIFA quintile (Q1 = most disadvantaged):")
display(_seg_table(tn, "seifa_q"))

print("test_normal — Δ MAE by brand (top 8 by volume):")
top_brands = tn["stn_brand_canonical"].value_counts().head(8).index
display(_seg_table(tn[tn["stn_brand_canonical"].isin(top_brands)], "stn_brand_canonical")
        .sort_values("Δ_MAE"))

print("Full segmentation (metro/regional, all brands, fuel) → results/comparison.md")

## 7. Residual diagnostics

Residuals over time across the v2.x folds. The "crisis-period blowup" the original notebook was set up to flag is real — Model A's residuals on test_crisis (now folder 6 in v3.0) are wider — but v3.0 absorbs this into the rotating k-fold rather than treating it as a separate concept.


In [ ]:
# Residual diagnostics. Residual = prediction − actual.
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# (a) Model B residual distribution per fold (clipped for readability)
for fold, df in preds.items():
    res = (df["y_pred_b"] - df["y_true"]).clip(-30, 30)
    axes[0].hist(res, bins=60, alpha=0.5, density=True, label=f"{fold} (n={len(res):,})")
axes[0].axvline(0, color="black", lw=0.8)
axes[0].set_title("Model B residuals (clipped ±30 c/L)")
axes[0].set_xlabel("pred − actual (c/L)")
axes[0].set_ylabel("density")
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

# (b) Weekly mean abs error over the whole test span, A vs B — watch for a
# crisis-period blowup.
both = pd.concat(preds.values(), ignore_index=True)
both["week"] = both["date"].dt.to_period("W").dt.start_time
both["ae_a"] = (both["y_pred_a"] - both["y_true"]).abs()
both["ae_b"] = (both["y_pred_b"] - both["y_true"]).abs()
wk = both.groupby("week")[["ae_a", "ae_b"]].mean()
axes[1].plot(wk.index, wk["ae_a"], lw=1.3, label="Model A")
axes[1].plot(wk.index, wk["ae_b"], lw=1.3, label="Model B")
axes[1].axvline(pd.Timestamp(config.TEST_CRISIS_START), color="red", ls="--",
                alpha=0.6, label="crisis start")
axes[1].set_title("Weekly mean abs error across the test span")
axes[1].set_ylabel("MAE (c/L)")
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Both models' error rises sharply at the 2026 crisis boundary (expected — "
      "OOD regime). Model B tracks at or below Model A throughout.")

## 8. Artifact provenance

Where the loaded artifacts come from (the notebook reads, it does not write — spec §9).


In [ ]:
# Artifacts are produced by the pipeline, not this notebook:
#
#   make train          -> models/model_a.pkl, models/model_b.pkl,
#                          models/feature_lists.json,
#                          models/predictions_test_{normal,crisis}.parquet
#   make evaluate       -> results/comparison.md  (v2.x single-split headline)
#   make train-kfold    -> models_kfold/fold_{1..6}/ + kfold_audit.json
#   make evaluate-kfold -> results/v3_phase1_smoke_kfold.md (or similar)
#
# v3.0 canonical artifacts under results/:
#   v3_phase2_summary.md                            -- 8-variant summary
#   v3_phase2_pr_b_baseline_kfold.md                -- PR B baseline k-fold report
#   v3_phase2_pr_c_*_kfold.md                       -- per-variant reports
#   v3_phase3_rank_consistency.md                   -- postmortem #1
#   v3_phase3_seed_noise_summary.md                 -- postmortem #2
#   v3_phase3_e6_seifa_dof_interaction_headline.md  -- postmortem #3
#   v3_phase3_hyperopt_summary.md                   -- postmortem #4 (sweep)
#   v3_phase3_hyperopt_validation.md                -- postmortem #4 (6-seed validation)
#
# Closing summary doc:
#   docs/research/2026-06_v3.0_phase3_closing_summary.md
#
print("Loaded artifacts:")
for p in sorted([
    Path("models/model_a.pkl"),
    Path("models/model_b.pkl"),
    Path("models/feature_lists.json"),
    Path("models/predictions_test_normal.parquet"),
    Path("models/predictions_test_crisis.parquet"),
]):
    if p.exists():
        size_kb = p.stat().st_size / 1024
        print(f"  {p}  ({size_kb:,.0f} KB)")
    else:
        print(f"  {p}  (missing — run `make train`)")
print()
print("v3.0 canonical headline lives at:")
print("  results/v3_phase2_pr_b_baseline_kfold.md")
print("  docs/research/2026-06_v3.0_phase3_closing_summary.md")
